rule generate + filtering + interpretation
exports: final rules

In [2]:
# import library
import ast
import pandas as pd
from mlxtend.frequent_patterns import association_rules

In [3]:
# Charge the frequent itemsets csv
itemsets = pd.read_csv("../outputs/frequent_itemsets.csv")

In [4]:
# Converting itemsets column to frozenset that can't be modify
def to_frozenset(x):
    if isinstance(x, frozenset):
        return x
    if pd.isna(x):
        return frozenset()
    s = str(x).strip()

    # make sure no error by gerer strings like "frozenset({1, 2})"
    if s.startswith("frozenset("):
        s = s[len("frozenset("):].rstrip(")").strip()

    # parsing
    parsed = ast.literal_eval(s)

    if isinstance(parsed, (set, list, tuple)):
        return frozenset(parsed)
    else:
        return frozenset([parsed])

itemsets["itemsets"] = itemsets["itemsets"].apply(to_frozenset)

print("itemsets shape:", itemsets.shape)
display(itemsets.head())

itemsets shape: (1467, 3)


,support,itemsets,length
0,0.002098,(34),1
1,0.005910,(45),1
2,0.011488,(196),1
3,0.002045,(248),1
4,0.007671,(260),1


In [5]:
# Create all rules it keep all rules
rules_all = association_rules(itemsets, metric="confidence", min_threshold=0.1)

print("rules_all:", rules_all.shape)
display(rules_all.head())

#filter simple rules, only good quality rules that efficace
MIN_CONFIDENCE = 0.2 #
MIN_LIFT = 1.1  #lift > 1 mean buy A make B more possible to be buy

rules_simple = rules_all[
    (rules_all["antecedents"].apply(lambda s: len(s) == 1)) &
    (rules_all["consequents"].apply(lambda s: len(s) == 1)) &
    (rules_all["confidence"] >= MIN_CONFIDENCE) &
    (rules_all["lift"] >= MIN_LIFT)
].copy()

print("rules_simple:", rules_simple.shape)

rules_all: (546, 14)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(432),(24852),0.009929,0.151680,0.002759,0.277841,1.831752,1.0,0.001253,1.174699,0.458628,0.017367,0.148718,0.148014
1,(1463),(13176),0.008985,0.121793,0.002174,0.241998,1.986960,1.0,0.001080,1.158581,0.501222,0.016907,0.136875,0.129925
2,(15290),(2295),0.012578,0.008172,0.002287,0.181790,22.246514,1.0,0.002184,1.212193,0.967215,0.123846,0.175049,0.230806
3,(2295),(15290),0.008172,0.012578,0.002287,0.279822,22.246514,1.0,0.002184,1.371081,0.962918,0.123846,0.270648,0.230806
4,(2295),(24852),0.008172,0.151680,0.002143,0.262304,1.729323,1.0,0.000904,1.149959,0.425214,0.013591,0.130403,0.138218


rules_simple: (179, 14)


In [6]:
#conversion of frozensetinto list for the export
rules_simple["antecedents"] = rules_simple["antecedents"].apply(lambda s: list(s))
rules_simple["consequents"] = rules_simple["consequents"].apply(lambda s: list(s))

rules_filtered = rules_simple[["antecedents", "consequents", "support", "confidence", "lift"]].copy()

display(rules_filtered.sort_values(["confidence", "lift"], ascending=False).head(10))

,antecedents,consequents,support,confidence,lift
23,[4957],[33754],0.002995,0.449513,46.849577
478,[36865],[28465],0.002319,0.441106,71.365326
498,[33787],[33754],0.002572,0.402552,41.955094
402,[41787],[24852],0.004392,0.386355,2.547169
375,[28204],[24852],0.010895,0.378693,2.496652
479,[28465],[36865],0.002319,0.375136,71.365326
360,[24799],[28465],0.002135,0.366718,59.330256
396,[39408],[24852],0.002815,0.364153,2.400797
417,[45066],[24852],0.009118,0.356128,2.347888
359,[28465],[24799],0.002135,0.345485,59.330256


In [7]:
#save the rule into outputs
rules_all.to_csv("../outputs/rules_all.csv", index=False)
rules_filtered.to_csv("../outputs/rules_filtered.csv", index=False)

In [ ]:
#justify the choice of confidence at 0.2

display(rules_all["confidence"].describe())
display(rules_filtered["confidence"].describe())

count    546.000000
mean       0.182326
std        0.061456
min        0.100343
25%        0.134537
50%        0.171042
75%        0.216749
max        0.449513
Name: confidence, dtype: float64

count    179.000000
mean       0.254160
std        0.047729
min        0.201064
25%        0.217529
50%        0.242002
75%        0.279541
max        0.449513
Name: confidence, dtype: float64

count    179.000000
mean       5.603681
std       12.351534
min        1.328606
25%        1.666098
50%        1.942079
75%        2.549669
max       71.365326
Name: lift, dtype: float64